# K-Nearest Neighbours
Accompanies the notes on **K-Nearest Neighbours**.

## 1. The hand example

In [ ]:
import numpy as np
from sklearn.neighbors import KNeighborsClassifier

pts = np.array([[1, 2], [2, 3], [3, 1], [5, 4], [6, 5]]); labels = np.array([0, 0, 0, 1, 1])
q = np.array([[3, 3]])
print("Euclidean distances:", np.linalg.norm(pts - q, axis=1).round(4))
print("Manhattan distances:", np.abs(pts - q).sum(axis=1))
for w in ["uniform", "distance"]:
    knn = KNeighborsClassifier(n_neighbors=3, weights=w).fit(pts, labels)
    dist, idx = knn.kneighbors(q)
    print(f"{w:8s}: neighbours {'ABCDE'[idx[0][0]]}{'ABCDE'[idx[0][1]]}{'ABCDE'[idx[0][2]]}, P(class 0) = {knn.predict_proba(q)[0, 0]:.4f}")

## 2. Distance concentration in high dimensions

In [ ]:
rng = np.random.default_rng(0)
for d in [2, 10, 100, 1000]:
    X = rng.random((1000, d))                        # 1000 points spread uniformly in [0, 1]^d
    query = rng.random(d)
    dist = np.linalg.norm(X - query, axis=1)
    print(f"d={d:>4}: nearest {dist.min():6.3f}  farthest {dist.max():6.3f}  (farthest-nearest)/nearest = {(dist.max() - dist.min()) / dist.min():.3f}")

## 3. Bank data: three ways of preparing the features

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

bd = pd.read_csv("../../data/bd_train.csv").drop(columns=["REF_NO", "post_code", "post_area"])
y = (bd.pop("Revenue.Grid") == 1).astype(int)
numeric = bd.select_dtypes("number").columns.tolist()
X = pd.get_dummies(bd, drop_first=True, dtype=float)
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=1, stratify=y)

param_grid = {"knn__n_neighbors": [5, 11, 21, 41, 81, 161], "knn__weights": ["uniform", "distance"], "knn__p": [1, 2]}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
versions = {"scale every column": StandardScaler(),
            "scale numeric, dummies 0/1": ColumnTransformer([("num", StandardScaler(), numeric)], remainder="passthrough"),
            "no scaling": "passthrough"}
for name, pre in versions.items():
    grid = GridSearchCV(Pipeline([("pre", pre), ("knn", KNeighborsClassifier())]), param_grid, cv=cv,
                        scoring="roc_auc", n_jobs=-1).fit(x_train, y_train)
    best = {k.replace("knn__", ""): v for k, v in grid.best_params_.items()}
    print(f"{name:27s}: best {best}, CV AUC {grid.best_score_:.4f}, "
          f"test AUC {roc_auc_score(y_test, grid.predict_proba(x_test)[:, 1]):.4f}")

## 4. Numeric columns only

In [ ]:
grid = GridSearchCV(Pipeline([("pre", StandardScaler()), ("knn", KNeighborsClassifier())]), param_grid, cv=cv,
                    scoring="roc_auc", n_jobs=-1).fit(x_train[numeric], y_train)
print(f"16 numeric columns only, scaled: best K {grid.best_params_['knn__n_neighbors']}, "
      f"test AUC {roc_auc_score(y_test, grid.predict_proba(x_test[numeric])[:, 1]):.4f}")

## 5. KNN regression on the loan data

In [ ]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_absolute_error

ld = pd.read_csv("../../data/loan_data_train.csv")
ld["Interest.Rate"] = pd.to_numeric(ld["Interest.Rate"].str.replace("%", ""), errors="coerce")
ld["FICO"] = ld["FICO.Range"].str.split("-", expand=True).astype(float).mean(axis=1)
ld["Loan.Length"] = pd.to_numeric(ld["Loan.Length"].str.replace(" months", ""), errors="coerce")
cols = ["FICO", "Loan.Length", "Amount.Requested", "Inquiries.in.the.Last.6.Months",
        "Monthly.Income", "Open.CREDIT.Lines", "Revolving.CREDIT.Balance"]
for col in cols:
    ld[col] = pd.to_numeric(ld[col], errors="coerce")
ld = ld[cols + ["Interest.Rate"]].dropna()
xr_train, xr_test, yr_train, yr_test = train_test_split(ld[cols], ld["Interest.Rate"], test_size=0.2, random_state=2)
for k in [1, 5, 15, 30, 60]:
    reg = Pipeline([("scale", StandardScaler()), ("knn", KNeighborsRegressor(n_neighbors=k))]).fit(xr_train, yr_train)
    print(f"K={k:>2}: train MAE {mean_absolute_error(yr_train, reg.predict(xr_train)):.3f}"
          f"   test MAE {mean_absolute_error(yr_test, reg.predict(xr_test)):.3f}")

## Try it yourself
1. Manhattan distance (`p=1`) in section 1.
2. KNN with only the 5 most important bank features.